In [ ]:
import os
import glob
from tqdm.auto import tqdm
import pandas as pd
import json
from langchain.docstore.document import Document as LangchainDocument
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()
from huggingface_hub import login

from langchain.chat_models import init_chat_model
import random
from langchain_community.vectorstores.utils import DistanceStrategy

from lib.models.embedder import Embedder
from lib.clients.llm import SimpleLLMFactory


/Users/sergey/Desktop/LangGraph-DIY-car-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
factory = SimpleLLMFactory(temperature=0)
gpt_oss_120b = factory.create("openai/gpt-oss-120b")

In [2]:
# Загружаем эмбеддер
embedder = Embedder()  # можно передать model_name="..." при желании
embeddings = embedder.embeddings  # это HuggingFaceEmbeddings из LangChain

1.1 Подготовка документов

Первая задача — разбить длинные статьи из базы знаний на удобные фрагменты. Для этого используется RecursiveCharacterTextSplitter. Этот класс разбивает текст на куски размером примерно 2 000 символов с 200‑символьным перекрытием; используются разные разделители (двойной и одинарный перенос строки, точка, пробел).

Разделение на куски позволяет создать семантически осмысленные отрывки, которые подходят для последующей индексации. Важно выбирать размер кусков не слишком маленьким (иначе могут потеряться контексты) и не слишком большим (чтобы не разбавлять смысл различными темами). В документации отмечено, что для измерения длины куска предпочтительнее считать токены, а не символы, поскольку векторные модели работают с токенами ￼.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Загрузка данных и разбиение на чанки
# ---------------------------------------------------------------------------
md_folder = "/Data/raw/articles"

def load_md_documents(docs_dir: str):
    docs = []
    for filepath in glob.glob(os.path.join(docs_dir, "*.md")):
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append(
            LangchainDocument(
                page_content=text, 
                metadata={"source": os.path.basename(filepath)}
            )
        )
    return docs

def split_documents(
    raw_docs: list,
    chunk_size: int,
    chunk_overlap: int,
    separators=None,
):
    if separators is None:
        separators = ["====", "\n\n", "\n", ".", " ", ""]
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
        separators=separators,
    )
    split_docs = []
    for doc in raw_docs:
        split_docs += text_splitter.split_documents([doc])
    unique = {}
    unique_docs = []
    for d in split_docs:
        if d.page_content not in unique:
            unique[d.page_content] = True
            unique_docs.append(d)
    return unique_docs

# Загрузка и разбиение
raw_docs = load_md_documents(md_folder)
docs_unique = split_documents(raw_docs, chunk_size=2000, chunk_overlap=200)

1.2 Генерация вопросов и ответов

Чтобы построить набор вопросов, используется модель gpt_oss_120b через API OPENROUTER. Задаётся шаблон подсказки (QA_generation_prompt), который предписывает модели формулировать фактологический вопрос и короткий ответ, не упоминая контекст напрямую.

Для каждого случайно выбранного отрывка из базы знаний модель синтезирует пару «вопрос–ответ». Для реальной оценки рекомендуется создать свыше 200, поскольку часть вопросов будет отфильтрована критическими агентами ￼.

In [ ]:
# ---------------------------------------------------------------------------
# 2. Генерация вопросов и ответов (QA-пары)
# ---------------------------------------------------------------------------
def call_llm_chat(model, prompt: str) -> str:
    response = model.invoke(prompt)
    return response.content  # возвращаем только текст

QA_generation_prompt = """
Ваша задача — сгенерировать **фактический вопрос** и **фактический ответ**, строго опираясь на предоставленный контекст.

### Языковые правила:
- И вопрос и ответ должны быть строго на  **Русском языке**.
- НЕ ИСПОЛЬЗУЙ другие языки.

### Требования к вопросу:
- Он ДОЛЖЕН быть ответимым с помощью конкретной, краткой фактической детали из контекста.
- Он ДОЛЖЕН быть сформулирован в стиле запроса для поисковой системы (четко, напрямую, естественно).
- Он НЕ ДОЛЖЕН ссылаться на существование контекста (например, избегайте фраз «согласно тексту», «на основе этого контекста» и т.п.).
- Он ДОЛЖЕН быть полностью понятен без дополнительного объяснения.

### Требования к ответу:
- Он ДОЛЖЕН быть фактическим ответом, взятым строго из контекста и ПОЛНОСТЬЮ отвечать на поставленный вопрос.
- Никаких рассуждений, только факт.
- Никакой выдуманной информации.

### Правила формата вывода (ОБЯЗАТЕЛЬНО):
Вы ДОЛЖНЫ вернуть **ТОЛЬКО действительный JSON** со следующей структурой:

{{
  "question": "string",
  "answer": "string"
}}

- НИКАКИХ markdown.
- НИКАКИХ комментариев.
- НИКАКИХ дополнительных текстов вне JSON.
- НИКАКИХ объяснений.
- НИКАКИХ строк до или после JSON.

### Контекст:
{context}
"""

# Генерируем вопросы/ответы

N_GENERATIONS = 200  # количество контекстов, на которых будем генерировать вопросы
generated_pairs = []

for sampled_doc in tqdm(random.sample(docs_unique, N_GENERATIONS), desc="Generating QA"):
    out = call_llm_chat(
        gpt_oss_120b,
        QA_generation_prompt.format(context=sampled_doc.page_content)
    )

    print("\nRAW OUTPUT:\n", out)  # вывод для диагностики

    try:
        data = json.loads(out)
        q = data["question"].strip()
        a = data["answer"].strip()

        generated_pairs.append({
            "context": sampled_doc.page_content,
            "question": q,
            "answer": a,
            "source_doc": sampled_doc.metadata["source"],
        })

    except Exception as e:
        print("JSON parse error:", e)
        continue

In [15]:
generated_pairs

[{'context': 'Ссылки\n- https://sappo.ru/blog/sam-sebe-detailer/khimchistka-salona-avtomobilya-svoimi-rukami/<https:/sappo.ru/product/mu_multi_cleaner_universalnyy_ochistitel_1_l_detail/\n- https://sappo.ru/blog/sam-sebe-detailer/khimchistka-salona-avtomobilya-svoimi-rukami/<https:/sappo.ru/product/ochistitel_alkantary_i_tkanevykh_poverkhnostey_pol_star_1l_koch_chemie/\n- https://sappo.ru/blog/sam-sebe-detailer/khimchistka-salona-avtomobilya-svoimi-rukami/<https:/sappo.ru/product/professionalnye_pyatnovyvoditeli_dlya_tekstilya_bond_1_2_3_goodmix/\n- https://sappo.ru/blog/sam-sebe-detailer/khimchistka-salona-avtomobilya-svoimi-rukami/<https:/sappo.ru/product/mehrzweckreiniger_universalnoe_moyushchee_sredstvo_1l_koch_chemie/\n- https://sappo.ru/blog/sam-sebe-detailer/khimchistka-salona-avtomobilya-svoimi-rukami/<https:/sappo.ru/product/mehrzweckreiniger_universalnoe_moyushchee_sredstvo_11_l_koch_chemie_11_kg/',
  'question': 'Какой объём у продукта Mehrzweckreiniger универсальное моющее 

1.3 Оценка и фильтрация вопросов

Сгенерированные вопросы могут содержать ошибки: они могут не соответствовать контексту, быть нерелевантными для разработчиков или зависеть от конкретного документа. Поэтому в ноутбуке реализованы три критических агента, каждый из которых оценивает вопрос по 5‑балльной шкале:

	1.	Приземлённость (groundedness) — проверяет, можно ли однозначно ответить на вопрос, используя только предоставленный контекст ￼.

	2.	Релевантность (relevance) — оценивает, насколько вопрос полезен для разработчиков машинного обучения, работающих с экосистемой Hugging Face ￼.
	
	3.	Самодостаточность (stand‑alone) — измеряет, понятен ли вопрос без дополнительной информации; ссылки на «этот документ» или «данный раздел» приводят к низкой оценке ￼.

Каждый агент сначала формирует текстовое обоснование, затем выдаёт числовую оценку. Пары «вопрос–ответ» с низкой оценкой по любому критерию (менее 4 из 5) удаляются из набора.

После фильтрации готовится итоговый синтетический датасет.

In [16]:
# ---------------------------------------------------------------------------
# 3. Критика и фильтрация QA-пар
# ---------------------------------------------------------------------------
# Промпты-критики
question_groundedness_critique_prompt = """
Вам будет дан контекст и вопрос.
Ваша задача — дать «общую оценку», определяющую, насколько хорошо можно однозначно ответить на данный вопрос по этому контексту.
Дайте оценку по шкале от 1 до 5, где 1 означает, что на вопрос невозможно ответить по контексту, а 5 означает, что вопрос можно ясно и недвусмысленно ответить по контексту.

Предоставьте ваш ответ в следующем виде:

Answer:::
Evaluation: (ваше обоснование оценки в виде текста)
Total rating: (ваша оценка, число от 1 до 5)

Теперь приведены вопрос и контекст.

Question: {question}\n
Context: {context}\n
Answer::: """

question_relevance_critique_prompt = """
Вам будет дан вопрос.
Ваша задача — дать «общую оценку», отражающую, насколько этот вопрос может быть полезен пользователям.
Дайте оценку по шкале от 1 до 5, где 1 означает, что вопрос совсем не полезен, и 5 означает, что вопрос чрезвычайно полезен.

Предоставьте ваш ответ в следующем виде:

Answer:::
Evaluation: (ваше обоснование оценки в виде текста)
Total rating: (ваша оценка, число от 1 до 5)

Теперь приведён вопрос.

Question: {question}\n
Answer::: """

question_standalone_critique_prompt = """
Вам будет дан вопрос.
Ваша задача — дать «общую оценку», отражающую, насколько вопрос независим от контекста.
Дайте оценку по шкале от 1 до 5, где 1 означает, что для понимания вопроса нужна дополнительная информация, и 5 означает, что вопрос понятен сам по себе.

Предоставьте ваш ответ в следующем виде:

Answer:::
Evaluation: (ваше обоснование оценки в виде текста)
Total rating: (ваша оценка, число от 1 до 5)

Теперь приведён вопрос.

Question: {question}\n
Answer::: """

# Фильтрация на основе оценок
filtered_pairs = []

for pair in tqdm(generated_pairs, desc="Filtering QA"):
    # Получаем ответы от модели в виде строки
    groundedness_response = call_llm_chat(
        gpt_oss_120b,
        question_groundedness_critique_prompt.format(
            context=pair["context"], question=pair["question"]
        ),
    )
    relevance_response = call_llm_chat(
        gpt_oss_120b,
        question_relevance_critique_prompt.format(question=pair["question"]),
    )
    standalone_response = call_llm_chat(
        gpt_oss_120b,
        question_standalone_critique_prompt.format(question=pair["question"]),
    )

    try:
        # Извлекаем числа после "Total rating:"
        g_score = int(groundedness_response.split("Total rating: ")[-1].strip())
        r_score = int(relevance_response.split("Total rating: ")[-1].strip())
        s_score = int(standalone_response.split("Total rating: ")[-1].strip())

        # Фильтруем по порогу
        if g_score >= 4 and r_score >= 4 and s_score >= 4:
            pair["groundedness_score"] = g_score
            pair["relevance_score"] = r_score
            pair["standalone_score"] = s_score
            filtered_pairs.append(pair)
    except Exception:
        # Если что-то не разобралось — пропускаем пару
        continue

# Сохраняем отфильтрованный датасет
eval_df = pd.DataFrame(filtered_pairs)
eval_df.to_csv("eval_dataset.csv", index=False, encoding="utf-8")

print(f"Готовый датасет сохранён: {len(eval_df)} примеров, файл eval_dataset.csv")

Filtering QA: 100%|██████████| 200/200 [49:56<00:00, 14.98s/it]

Готовый датасет сохранён: 69 примеров, файл eval_dataset.csv


In [17]:
eval_df

,context,question,answer,source_doc,groundedness_score,relevance_score,standalone_score
0,Полировка фар своими руками — 4\nТеперь оптика...,Какой абразивности должны быть шлифовальные ли...,800-1000 и 1500-2000,kak-otpolirovat-fary_cleaned.md,5,4,4
1,====\nКак ухаживать за микрофибрами после рабо...,Какой режим стирки рекомендуется для микрофибр...,Деликатный режим стирки,mikrofibry-dlya-chego-nuzhny-kakie-byvayut-kak...,5,4,5
2,После этого остается лишь располировать средст...,Какой продукт рекомендуется использовать для п...,Метализированную вату от американского бренда ...,kak-pravilno-chistit-i-ukhazhivat-za-khromirov...,5,4,5
3,Отполировать можно только неглубокие царапины....,Какой инструмент рекомендуется использовать дл...,Наждачная абразивная бумага от 1500 до 3000,kakie-carapiny-mozhno-ustranit-s-pomoshyu-poli...,5,4,5
4,====\nО принципах удаления пятен\nЕсли неправи...,Какой материал рекомендуется использовать для ...,Микрофибра,kak-otmyt-bitum-s-kuzova-avtomobilya_cleaned.md,5,4,5
...,...,...,...,...,...,...,...
64,"Нельзя, конечно, исключать такую возможность. ...",Что следует укрыть полиэтиленовыми пакетами дл...,Блок предохранителей и ЭБУ двигателя,moyka-dvigatelya-zachem-i-kak-chasto-neobkhodi...,5,4,5
65,====\nОсновные правила чистки сидений\n- До то...,Что следует сделать перед началом самостоятель...,Протестировать выбранное средство на небольшом...,chistka-sidenij-avtomobilya-osnovnye-pravila-i...,5,4,5
66,Химчистка салона автомобиля своими руками\nЕще...,Что понадобится для химчистки салона автомобил...,"Микрофибровые полотенца, чистая ветошь, тканев...",khimchistka-salona-avtomobilya-svoimi-rukami_c...,5,4,5
67,====\nЭтапы полировки кузова автомобиля \nСост...,Каким способом удаляют остатки пасты с кузова ...,Выбивают водой под давлением.,chto-takoe-polirovka-avtomobilya_cleaned.md,5,4,5
